# 0. ANCHOR Classical Planner Benchmark

ANCHOR Alpha is a deterministic, scientifically constrained research-and-education sandbox for investigating adaptive underwater-glider mission planning. It supports reproducible comparison of human, classical, and learning-based planners. It is not an operational ocean forecast or certified vehicle-navigation system.

**Plan. Simulate. Compare. Learn.**

This notebook proposes and visualizes plans outside the browser. It does not own mission validation, simulation, or official scoring.

Colab proposes. ANCHOR validates. ANCHOR simulates. ANCHOR scores.

Scientific boundary: these fixtures are deterministic synthetic benchmark data. They are suitable for reproducible planner comparison and classroom analysis, not calibrated ocean forecasting or certified navigation.

Fairness boundary: the default workflow is `FORECAST_ONLY`. Supported fairness classes are `FORECAST_ONLY`, `BELIEF_AWARE`, `PUBLIC_OBSERVATION_ONLY`, and explicit opt-in `ORACLE_HIDDEN_TRUTH`. Hidden truth is excluded unless an explicit oracle/debug workflow is selected and labeled.

Optimality boundary: An exact result is exact only for the stated candidate set, state representation, objective, and discretization. The Exact Bounded Small-Instance Oracle is bounded by declared candidates, objective, state representation, and discretization.

## 1. Configuration

Choose where the solver packet comes from and which transparent planners to run. No local absolute paths are required.

In [ ]:
DATA_ACQUISITION_MODE = "checked_in_fixture"  # checked_in_fixture | upload | static_url
SOLVER_PACKET_PATH = "tests/fixtures/colab_benchmark/static_additive_routing_solver_packet.json"
STATIC_DOWNLOAD_BASE_URL = ""  # optional, e.g. a GitHub Pages base URL ending before the fixture path
BENCHMARK_FIXTURE_ID = "static_additive_routing"
OUTPUT_DIR = "anchor_benchmark_output"
PLANNER_SEED = 7
ACTIVE_GLIDER = "glider_01"
FAIRNESS_CLASS = "FORECAST_ONLY"
ALLOW_ORACLE_HIDDEN_TRUTH = False
CANDIDATE_NODE_LIMIT = 24
EXACT_ORACLE_SIZE_LIMIT = 6
REPEAT_COUNT = 3
TIMEOUT_PER_PLANNER_SECONDS = 5.0
PROFILE_POLICY = "mission/default"
PLANNERS = ["dijkstra", "astar", "weightedAstar", "greedyValuePerCost", "beamSearch", "timeExpandedAstar"]
RUN_EXACT_ORACLE = True
PLOT_FIGURES = True
RUN_NODE_REFEREE = True

## 2. Environment Setup

The core planners use the Python standard library. `pandas` and `matplotlib` are optional notebook conveniences. Node is used only to call the authoritative ANCHOR validator, simulator, and scorer when the repository is available.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
TOOLS_PYTHON = REPO_ROOT / "tools" / "python"
if TOOLS_PYTHON.exists():
    sys.path.insert(0, str(TOOLS_PYTHON))

from anchor_benchmark import (
    astar_search,
    beam_search,
    build_anchor_plan,
    build_benchmark_record,
    build_colab_acceptance_report,
    build_parity_table,
    build_planning_problem,
    build_reproducibility_manifest,
    compare_results,
    dijkstra_search,
    exact_small_instance_oracle,
    extract_public_environment,
    greedy_value_per_cost,
    load_json,
    run_planner_suite,
    sample_public_environment,
    stable_digest,
    summarize_public_environment,
    time_expanded_astar,
    validate_parity_probes,
    validate_solver_packet,
    weighted_astar_search,
    write_json,
)
from anchor_benchmark.io import BENCHMARK_BOUNDARY, NOTEBOOK_VERSION, node_version, python_runtime_summary

try:
    import pandas as pd
except Exception:
    pd = None
try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

output_root = Path(OUTPUT_DIR)
for name in ["plans", "results", "benchmark_records", "figures", "tables"]:
    (output_root / name).mkdir(parents=True, exist_ok=True)

print("Notebook version:", NOTEBOOK_VERSION)
print("Boundary:", BENCHMARK_BOUNDARY)
print("Python:", python_runtime_summary())
print("Node:", node_version())
print("pandas:", getattr(pd, "__version__", "unavailable"))
print("matplotlib:", getattr(plt, "__version__", "available" if plt else "unavailable"))

## 3. Obtain Benchmark Data

Supported paths: upload an exported solver packet, load a checked-in deterministic fixture, or fetch a public fixture from a user-configured static base URL. The default does not require network access.

In [ ]:
def load_solver_packet():
    if DATA_ACQUISITION_MODE == "checked_in_fixture":
        return load_json(SOLVER_PACKET_PATH)
    if DATA_ACQUISITION_MODE == "upload":
        try:
            from google.colab import files
            uploaded = files.upload()
            first_name = next(iter(uploaded))
            return json.loads(uploaded[first_name].decode("utf-8"))
        except Exception as exc:
            raise RuntimeError("Upload mode requires Google Colab file upload support.") from exc
    if DATA_ACQUISITION_MODE == "static_url":
        if not STATIC_DOWNLOAD_BASE_URL:
            raise ValueError("STATIC_DOWNLOAD_BASE_URL is required for static_url mode.")
        from urllib.request import urlopen
        url = STATIC_DOWNLOAD_BASE_URL.rstrip("/") + "/" + SOLVER_PACKET_PATH
        with urlopen(url, timeout=15) as handle:
            return json.loads(handle.read().decode("utf-8"))
    raise ValueError(f"Unknown DATA_ACQUISITION_MODE: {DATA_ACQUISITION_MODE}")

solver_packet = load_solver_packet()
print("Loaded:", solver_packet.get("packetId"), "digest", stable_digest(solver_packet))

## 4. Validate and Inspect Artifacts

Hard failures stop benchmark execution. This notebook-side validation checks metadata and visibility; codec and route execution validation still belong to ANCHOR.

In [ ]:
validation = validate_solver_packet(solver_packet, allow_oracle=ALLOW_ORACLE_HIDDEN_TRUTH)
print(json.dumps(validation, indent=2))
if not validation["ok"]:
    raise RuntimeError("Solver packet validation failed before planning.")

artifact_overview = {
    "artifactKind": solver_packet.get("type"),
    "schemaVersion": solver_packet.get("schemaVersion"),
    "solverPacketDigest": validation["solverPacketDigest"],
    "environmentDigest": solver_packet.get("environmentDigest"),
    "missionDigest": solver_packet.get("missionDigest"),
    "coordinateFrame": solver_packet.get("coordinateFrame", "grid with cellSizeMeters"),
    "horizontalUnits": "meters via world.grid.cellSizeMeters",
    "depthConvention": "positive-down depth layers when present",
    "timeConvention": "seconds",
    "fieldRoles": list(((solver_packet.get("planningData") or {}).get("visibleFields") or {}).keys()),
    "visibilityClass": validation["visibilityClass"],
    "fairnessClass": validation["fairnessClass"],
    "scoreProfileId": solver_packet.get("scoreProfileId"),
    "scoreProfileVersion": solver_packet.get("scoreProfileVersion"),
}
print(json.dumps(artifact_overview, indent=2))

## 5. Scientific Validation Context

The SCI-VALID-R2A manifest documents deterministic package evidence, claim boundaries, and limitations. It is not proof of operational ocean forecast accuracy.

In [ ]:
manifest_path = Path("validation/manifest.json")
if manifest_path.exists():
    validation_manifest = load_json(manifest_path)
    validation_context = {
        "manifestId": validation_manifest.get("manifestId") or validation_manifest.get("id"),
        "manifestDigest": validation_manifest.get("manifestDigest") or validation_manifest.get("digest"),
        "statusSummary": validation_manifest.get("statusSummary"),
        "evidenceLevelSummary": validation_manifest.get("evidenceLevelSummary"),
        "benchmarkSuitabilitySummary": validation_manifest.get("benchmarkSuitabilitySummary"),
    }
else:
    validation_context = {"manifestId": "UNKNOWN", "manifestDigest": solver_packet.get("validationBaselineDigest"), "statusSummary": "manifest file not found in this runtime"}
print(json.dumps(validation_context, indent=2))

## 6. Visualize the Environment

Visualizations use solver-visible fields only by default. Integrated or oracle views must be labeled explicitly.

In [ ]:
problem = build_planning_problem(solver_packet, candidate_node_limit=CANDIDATE_NODE_LIMIT, profile_policy=PROFILE_POLICY)
print("Candidate count:", len(problem.candidates), "fairness:", problem.fairness_class)
if PLOT_FIGURES and plt is not None:
    from anchor_benchmark.visualization import plot_current_field, plot_environment_overview, plot_scalar_field
    figs = [plot_environment_overview(problem), plot_current_field(problem), plot_scalar_field(problem)]
    for index, fig in enumerate(figs, start=1):
        fig.savefig(output_root / "figures" / f"environment_{index:02d}.png", dpi=140, bbox_inches="tight")
        plt.show()
else:
    print("Plotting skipped; matplotlib unavailable or PLOT_FIGURES=False.")

## Exported Data Integrity and Web-App Parity

This section checks the data the notebook actually receives from ANCHOR's public benchmark export. Visual agreement is an inspection aid. Canonical digest and numerical sample agreement are the authoritative parity evidence.

The compact R1 fixtures expose forecast-visible terrain masks, hazards, current frames, scalar/ROI frames, mission geometry, candidates, and metadata. They do not expose hidden truth. Some fixtures expose terrain masks rather than calibrated bathymetry and surface-style forecast frames rather than full depth-resolved current/scalar cubes; those limitations are recorded as warnings instead of being hidden by Python-generated substitute fields.

In [ ]:
public_environment = extract_public_environment(solver_packet)
public_summary = summarize_public_environment(solver_packet)
parity_result = validate_parity_probes(solver_packet)
parity_table = build_parity_table(solver_packet, parity_result)

print("Public environment digest:", stable_digest(public_environment))
print("Axes:", public_summary["axes"])
print("Bathymetry status:", public_summary["bathymetry"])
print("Mask counts:", public_summary["masks"])
print("Current stats:", public_summary["currents"])
print("Scalar stats:", public_summary["scalars"])
print("Parity probes:", parity_result["status"], parity_result["probeCount"], "failed", parity_result["failedProbeCount"])
if parity_result["status"] == "FAIL":
    raise RuntimeError("Exported-data parity probes failed; do not continue to planner comparison.")

if pd is not None:
    display(pd.DataFrame(parity_table))
    display(pd.DataFrame(parity_result["rows"]))
else:
    print(json.dumps(parity_table, indent=2))
    print(json.dumps(parity_result["rows"], indent=2)[:4000])

write_json(output_root / "tables" / "public_environment_summary.json", public_summary)
write_json(output_root / "tables" / "parity_probe_results.json", parity_result)
write_json(output_root / "tables" / "parity_table.json", {"rows": parity_table})

## 7. Construct the Planning Problem

Candidate discretization constrains the search space. Search costs guide plan construction; official mission outcomes come from ANCHOR simulation and scoring.

In [ ]:
cost_terms = [term.__dict__ for term in problem.cost_terms]
candidate_table = [node.__dict__ for node in problem.candidates]
print("Cost terms:")
print(json.dumps(cost_terms, indent=2))
if pd is not None:
    display(pd.DataFrame(candidate_table))
else:
    print(json.dumps(candidate_table[:8], indent=2))

## 8. Run Classical Planners

Dijkstra and A* can be exact on the declared graph under their assumptions. Weighted A*, Greedy Value per Predicted Cost, and Beam Search are labeled heuristic. Time-Expanded A* is exact only for declared time bins when the assumptions hold.

In [ ]:
planner_results = run_planner_suite(problem, PLANNERS)
if RUN_EXACT_ORACLE and len(problem.candidates) <= EXACT_ORACLE_SIZE_LIMIT + 1:
    planner_results.append(exact_small_instance_oracle(problem, candidate_limit=EXACT_ORACLE_SIZE_LIMIT, route_depth=4))

dijkstra = next((r for r in planner_results if r.planner_id == "dijkstra"), None)
astar = next((r for r in planner_results if r.planner_id == "astar"), None)
if dijkstra and astar:
    print("Dijkstra/A* cost delta:", abs(dijkstra.cost - astar.cost))

search_rows = compare_results(planner_results)
if pd is not None:
    display(pd.DataFrame(search_rows))
else:
    print(json.dumps(search_rows, indent=2))

## 9. Export Candidate ANCHOR Plans

Waypoints remain horizontal destinations. Incoming segment/profile metadata describes behavior used to reach the destination.

In [ ]:
plans = {}
for result in planner_results:
    plan = build_anchor_plan(problem, result, agent_id=ACTIVE_GLIDER)
    path = output_root / "plans" / f"{result.planner_id}.anchor.plan.json"
    write_json(path, plan)
    plans[result.planner_id] = {"path": path, "plan": plan, "result": result}
    print("wrote", path, stable_digest(plan))

## 10. Validate Plans with ANCHOR

Notebook checks are not enough. When Node and the repository are available, call the canonical ANCHOR plan validator.

In [ ]:
def run_node_validate(plan_path):
    if shutil.which("node") is None or not Path("tools/js/headless_validate_plan.mjs").exists():
        return {"ok": None, "status": "SKIPPED", "reason": "Node or ANCHOR repo files unavailable"}
    completed = subprocess.run(["node", "tools/js/headless_validate_plan.mjs", SOLVER_PACKET_PATH, str(plan_path)], capture_output=True, text=True)
    payload = json.loads(completed.stdout) if completed.stdout.strip().startswith("{") else {"stdout": completed.stdout, "stderr": completed.stderr}
    payload["returncode"] = completed.returncode
    return payload

validation_reports = {}
for planner_id, entry in plans.items():
    validation_reports[planner_id] = run_node_validate(entry["path"])
    print(planner_id, validation_reports[planner_id].get("ok"), validation_reports[planner_id].get("returncode"))

## 11. Simulate and Score with the Authoritative Referee

Every official benchmark score must originate from the same ANCHOR referee used by browser, headless, and benchmark execution.

In [ ]:
def run_node_referee(planner_id, plan_path):
    if not RUN_NODE_REFEREE or shutil.which("node") is None or not Path("tools/js/evaluate_colab_benchmark_plan.mjs").exists():
        return {"ok": None, "status": "SKIPPED", "reason": "Node referee unavailable"}
    out_dir = output_root / "results" / planner_id
    cmd = ["node", "tools/js/evaluate_colab_benchmark_plan.mjs", "--solver-packet", SOLVER_PACKET_PATH, "--plan", str(plan_path), "--out", str(out_dir), "--agent-id", ACTIVE_GLIDER]
    completed = subprocess.run(cmd, capture_output=True, text=True)
    payload = json.loads(completed.stdout) if completed.stdout.strip().startswith("{") else {"stdout": completed.stdout, "stderr": completed.stderr}
    payload["returncode"] = completed.returncode
    return payload

official_results = {}
for planner_id, entry in plans.items():
    official_results[planner_id] = run_node_referee(planner_id, entry["path"])
    print(planner_id, official_results[planner_id].get("ok"), official_results[planner_id].get("finalScore"))

## 12. Compare Algorithms

Tables keep planner solve time separate from ANCHOR validation/simulation/scoring time. Forecast-only and oracle-assisted results should not be silently mixed.

In [ ]:
comparison_rows = []
for row in search_rows:
    official = official_results.get(row["plannerId"], {})
    comparison_rows.append({**row, "officialScore": official.get("finalScore"), "evaluationOk": official.get("ok"), "benchmarkRecordDigest": official.get("benchmarkRecordDigest")})

summary_path = output_root / "tables" / "benchmark_results.csv"
if pd is not None:
    df = pd.DataFrame(comparison_rows)
    display(df)
    df.to_csv(summary_path, index=False)
else:
    import csv
    with summary_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(comparison_rows[0].keys()))
        writer.writeheader()
        writer.writerows(comparison_rows)
print("wrote", summary_path)

## 13. Visualize Planned and Realized Outcomes

Planned routes are notebook artifacts. Realized trajectories, score results, and samples come from ANCHOR output bundles when available.

In [ ]:
if PLOT_FIGURES and plt is not None:
    from anchor_benchmark.visualization import plot_planner_routes
    fig = plot_planner_routes(problem, planner_results)
    fig.savefig(output_root / "figures" / "planner_routes.png", dpi=140, bbox_inches="tight")
    plt.show()
else:
    print("Route plot skipped.")

for planner_id, result in official_results.items():
    bundle = Path(result.get("files", {}).get("bundle", ""))
    if bundle.exists():
        print(planner_id, "bundle available:", bundle)

## 14. Export Benchmark Artifacts

The deterministic output structure is `anchor_benchmark_output/plans`, `results`, `benchmark_records`, `figures`, `tables`, plus summary and reproducibility files.

In [ ]:
records = []
for planner_id, entry in plans.items():
    official = official_results.get(planner_id, {})
    official_evaluation = {"officialScore": official.get("finalScore"), "scoreProfileId": official.get("scoreProfileId"), "totalEvaluationTimeSeconds": official.get("totalEvaluationTimeSeconds")}
    record = build_benchmark_record(problem, entry["result"], plan=entry["plan"], official_evaluation=official_evaluation)
    record_path = output_root / "benchmark_records" / f"{planner_id}.benchmark-record.json"
    write_json(record_path, record)
    records.append(record)

benchmark_summary = {"rows": comparison_rows, "solverPacketDigest": stable_digest(solver_packet), "boundary": BENCHMARK_BOUNDARY}
write_json(output_root / "benchmark_summary.json", benchmark_summary)
print("exported records:", len(records))

## 15. Reproducibility Summary

A planner's benchmark result is meaningful only with its environment digest, mission digest, fairness class, simulator version, scoring profile, and validation baseline.

In [ ]:
def git_commit():
    try:
        completed = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True)
        return completed.stdout.strip() or "UNKNOWN"
    except Exception:
        return "UNKNOWN"

generated_paths = [str(path) for path in output_root.rglob("*") if path.is_file()]
manifest = build_reproducibility_manifest(problem, records, repository_commit=git_commit(), python_version=sys.version.split()[0], node_version=node_version(), generated_paths=generated_paths)
write_json(output_root / "reproducibility_manifest.json", manifest)

acceptance_report = build_colab_acceptance_report(
    packet=solver_packet,
    validation_report=validation,
    parity_result=parity_result,
    parity_table=parity_table,
    algorithm_results=planner_results,
    official_results=official_results,
    output_artifacts=[str(path.relative_to(output_root)) for path in output_root.rglob("*") if path.is_file()],
    notebook_digest=stable_digest(json.loads(Path("tools/python/notebooks/anchor_classical_planner_benchmark.ipynb").read_text(encoding="utf-8"))) if Path("tools/python/notebooks/anchor_classical_planner_benchmark.ipynb").exists() else "UNKNOWN",
    repository_commit=git_commit(),
    library_versions={"pandas": getattr(pd, "__version__", "unavailable"), "matplotlib": getattr(plt, "__version__", "available" if plt else "unavailable")},
)
write_json(output_root / "colab_acceptance_report.json", acceptance_report)
print(json.dumps(manifest, indent=2)[:4000])
print(json.dumps({"acceptanceStatus": acceptance_report["status"], "reportDigest": acceptance_report["reportDigest"], "failures": acceptance_report["failures"]}, indent=2))
if acceptance_report["status"] == "PASS":
    print("COLAB-BENCH-R1.1 ACCEPTANCE: PASS")
else:
    print("COLAB-BENCH-R1.1 ACCEPTANCE: FAIL")

try:
    from google.colab import files
    print("In Colab: use files.download on selected output files or zip the output directory if needed.")
except Exception:
    print("Outside Colab, output artifacts are available under", output_root)